# Baseline Training — ResNet-20 on CIFAR-10

Vanilla SGD (no momentum) with a step-decay schedule, following the original ResNet paper setup.

In [2]:
import sys
from pathlib import Path
import torch
import torch.nn as nn

sys.path.append(str(Path.cwd().parent))



from src.data import get_loaders
from src.model import resnet20

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
train_loader, test_loader = get_loaders(batch_size=128)

len(train_loader), len(test_loader)

/Users/alexandre/anaconda3/envs/OptML/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


(391, 79)

In [4]:
model = resnet20().to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.0,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[82, 123],
    gamma=0.1
)

criterion = nn.CrossEntropyLoss()

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score

num_epochs = 164

for epoch in tqdm(range(num_epochs), desc="Training progress"):
    model.train()
    running_loss = 0.0
    train_preds, train_labels = [], []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = outputs.detach().max(1)
        train_preds.extend(predicted.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())
        running_loss += loss.item()

    scheduler.step()

    train_acc = accuracy_score(train_labels, train_preds) * 100
    train_f1  = f1_score(train_labels, train_preds, average="macro") * 100
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {running_loss/len(train_loader):.4f}  Acc: {train_acc:.2f}%  Macro F1: {train_f1:.2f}%")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc      = accuracy_score(all_labels, all_preds) * 100
macro_f1 = f1_score(all_labels, all_preds, average="macro") * 100

print(f"Accuracy : {acc:.2f}%")
print(f"Macro F1 : {macro_f1:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
fig.colorbar(im, ax=ax)
ax.set(
    xticks=range(len(CIFAR10_CLASSES)),
    yticks=range(len(CIFAR10_CLASSES)),
    xticklabels=CIFAR10_CLASSES,
    yticklabels=CIFAR10_CLASSES,
    xlabel="Predicted",
    ylabel="True",
    title="Confusion Matrix — ResNet-20 on CIFAR-10 (Baseline)",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for i in range(len(CIFAR10_CLASSES)):
    for j in range(len(CIFAR10_CLASSES)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
plt.show()